# Generative AI – Combined Beginner Notebook

This notebook combines the:

1. Text Summarizer
2. Sentiment Analysis
3. ROUGE Benchmark
4. Multi-PDF Question Answering / Retriever

## Compatible versions

- **OpenAI:** 0.28.1
- **LangChain:** 0.0.335

The notebook is intentionally simplified for beginners.




# Step 1: Check Versions and Import Libraries

Run this cell first.


In [ ]:
import openai
import langchain

print("OpenAI version:", openai.__version__)
print("LangChain version:", langchain.__version__)
print("Required OpenAI version: 0.28.1")
print("Required LangChain version: 0.0.335")


OpenAI version: 0.28.1
LangChain version: 0.0.335
Required OpenAI version: 0.28.1
Required LangChain version: 0.0.335


# Part 1 – Text Summarizer

We will:
1. Read a PDF
2. Extract the text
3. Count tokens
4. Send the text to an OpenAI model through LangChain
5. Generate a short summary

Place `arxiv_impact_of_GENAI.pdf` in the same folder as this notebook.


## Step 2: Import PDF and LangChain Tools

In [ ]:
from pypdf import PdfReader
import tiktoken

from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain


## Step 3: Read the PDF

In [ ]:
PAPER_PATH = "arxiv_impact_of_GENAI.pdf"

reader = PdfReader(PAPER_PATH)

print("Number of pages:", len(reader.pages))


Number of pages: 9


## Step 4: Extract Text

In [ ]:
# @title
text_body = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        text_body += text + "\n"

print(text_body[:3000])


The Impact of Generative Artificial Intelligence
Kaichen Zhang1,2, Ohchan Kwon2, Hui Xiong1,2
1Hong Kong University of Science and Technology (Guangzhou)
2Hong Kong University of Science and Technology
kzhangbi@connect.ust.hk, ohchankw@ust.hk, xionghui@ust.hk
Abstract
The rise of generative artificial intelligence (AI) has sparked
concerns about its potential influence on unemployment and
market depression. This study addresses this concern by ex-
amining the impact of generative AI on product markets. To
overcome the challenge of causal inference, given the inher-
ent limitations of conducting controlled experiments, this pa-
per identifies an unanticipated and sudden leak of a highly
proficient image-generative AI as a novel instance of a ”natu-
ral experiment.” This AI leak spread rapidly, significantly re-
ducing the cost of generating anime-style images compared to
other styles, creating an opportunity for comparative assess-
ment. We collect real-world data from an artwork outsou

## Step 5: Count Tokens

In [ ]:
encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
token_count = len(encoding.encode(text_body))

print("Token count:", token_count)


Token count: 10570


## Step 6: Generate a Summary




In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ =load_dotenv(find_dotenv())
openai.api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded")

API key loaded


In [ ]:
prompt = PromptTemplate(
    input_variables=["paper"],
    template="""Summarize the following research paper in simple language.

Include:
1. Main topic
2. Key findings
3. Methods
4. Important conclusion

Paper:
{paper}
"""
)

llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=0.2
)

summary_chain = LLMChain(
    llm=llm,
    prompt=prompt
)

summary = summary_chain.run(paper=text_body[:12000])

print(summary)


1. Main topic: The research paper explores the impact of generative artificial intelligence (AI) on product markets.
2. Key findings: The study shows that generative AI lowers average prices but significantly boosts order volume and overall revenue, benefiting artists and contributing to market prosperity.
3. Methods: The researchers used real-world data from an artwork outsourcing platform and identified an unexpected leak of a highly proficient image-generative AI as a natural experiment to study the impact of generative AI.
4. Important conclusion: The paper suggests that generative AI does not replace artists but instead helps increase their income and industry prosperity, with implications for practitioners, policymakers, and the AI community.


## Step 7: Save the Summary

The saved file can be used in the ROUGE section.


In [ ]:
with open("Summary.txt", "w", encoding="utf-8") as f:
    f.write(summary)

print("Summary saved as Summary.txt")


Summary saved as Summary.txt


# Part 2 – Sentiment Analysis

We compare two approaches:

- OpenAI
- TextBlob

TextBlob gives a polarity score between -1 and +1.


## Step 8: Import TextBlob

In [ ]:
from textblob import TextBlob


## Step 9: Sentiment Function

In [ ]:
def sentiment_analysis(text):
    prompt = f"""Classify the sentiment as Positive, Negative, or Neutral.

Text: {text}

Answer with only one word."""

    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    openai_sentiment = response.choices[0].message["content"].strip()

    polarity = TextBlob(text).sentiment.polarity
    textblob_sentiment = (
        "Positive" if polarity > 0 else
        "Negative" if polarity < 0 else
        "Neutral"
    )

    print("Text:", text)
    print("OpenAI:", openai_sentiment)
    print("TextBlob:", textblob_sentiment)
    print("Polarity:", polarity)

text = "I absolutely love this course. It is very useful."
sentiment_analysis(text)

Text: I absolutely love this course. It is very useful.
OpenAI: Positive
TextBlob: Positive
Polarity: 0.445


## Step 10: Try Sentiment Analysis

In [ ]:
text = "I really enjoyed this course. The explanations were clear and useful."

sentiment_analysis(text)


Text: I really enjoyed this course. The explanations were clear and useful.
OpenAI: Positive
TextBlob: Positive
Polarity: 0.3


# Part 3 – ROUGE Benchmark

ROUGE compares a generated summary with a reference text.

The original demo used `datasets.load_metric("rouge")`. That approach can create compatibility problems with newer `datasets` versions.




## Step 11: Read the Original PDF and Summary

In [ ]:
from collections import Counter

with open("Summary.txt", "r", encoding="utf-8") as f:
    summary_text = f.read()

reference_text = text_body

print("Summary characters:", len(summary_text))
print("Reference characters:", len(reference_text))


Summary characters: 758
Reference characters: 42533


## Step 12: Simple ROUGE-1 Calculation

In [ ]:
def rouge1(prediction, reference):
    pred_words = prediction.lower().split()
    ref_words = reference.lower().split()

    pred_count = Counter(pred_words)
    ref_count = Counter(ref_words)

    overlap = sum((pred_count & ref_count).values())

    precision = overlap / len(pred_words) if pred_words else 0
    recall = overlap / len(ref_words) if ref_words else 0

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1

precision, recall, f1 = rouge1(summary_text, reference_text)

print("ROUGE-1 Precision:", round(precision, 4))
print("ROUGE-1 Recall:", round(recall, 4))
print("ROUGE-1 F1:", round(f1, 4))


ROUGE-1 Precision: 0.9174
ROUGE-1 Recall: 0.0153
ROUGE-1 F1: 0.0302


### Interpretation

- **Precision:** How much of the generated summary overlaps with the reference.
- **Recall:** How much of the reference content is covered by the summary.
- **F1:** Combined measure of precision and recall.

A higher value indicates greater word overlap.


# Part 4 – Multi-PDF Question Answering

We will use a simpler approach:

`PDF → Chunks → Keyword Retrieval → Relevant Text`



## Step 13: Load PDFs from a Folder

Create a folder named `GenAI_Papers` and put your PDF files inside it.


In [ ]:
from pathlib import Path
from pypdf import PdfReader

pdf_folder = Path("GenAI_Papers")

documents = []

for pdf_file in pdf_folder.glob("*.pdf"):
    reader = PdfReader(str(pdf_file))

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text()

        if page_text:
            documents.append({
                "file": pdf_file.name,
                "page": page_number,
                "text": page_text
            })

print("PDF pages loaded:", len(documents))


PDF pages loaded: 142


## Step 14: Split Text into Small Chunks

In [ ]:
def create_chunks(text, chunk_size=800):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i + chunk_size]))

    return chunks

chunks = []

for doc in documents:
    for chunk in create_chunks(doc["text"]):
        chunks.append({
            "file": doc["file"],
            "page": doc["page"],
            "text": chunk
        })

print("Number of chunks:", len(chunks))


Number of chunks: 151


## Step 15: Simple Retriever

The retriever checks how many query words appear in each chunk and returns the most relevant chunk.


In [ ]:
def retrieve(query, chunks, top_k=2):
    query_words = set(query.lower().split())

    results = []

    for chunk in chunks:
        chunk_words = set(chunk["text"].lower().split())
        score = len(query_words & chunk_words)

        results.append((score, chunk))

    results.sort(key=lambda x: x[0], reverse=True)

    return [item[1] for item in results[:top_k]]

query = "What is toolformer?"

results = retrieve(query, chunks)

for result in results:
    print("File:", result["file"])
    print("Page:", result["page"])
    print(result["text"][:1500])
    print("-" * 60)


File: toolformer.pdf
Page: 1
Toolformer: Language Models Can Teach Themselves to Use Tools Timo Schick Jane Dwivedi-Yu Roberto Dessì†Roberta Raileanu Maria Lomeli Luke Zettlemoyer Nicola Cancedda Thomas Scialom Meta AI Research†Universitat Pompeu Fabra Abstract Language models (LMs) exhibit remarkable abilities to solve new tasks from just a few examples or textual instructions, especially at scale. They also, paradoxically, struggle with basic functionality, such as arithmetic or fac- tual lookup, where much simpler and smaller models excel. In this paper, we show that LMs can teach themselves to use external tools via simple APIs and achieve the best of both worlds. We introduce Toolformer , a model trained to decide which APIs to call, when to call them, what arguments to pass, and how to best incorporate the results into future token prediction. This is done in a self-supervised way, requiring nothing more than a handful of demonstrations for each API. We incorporate a range of too

## Step 16: Ask Another Question

In [ ]:
query = input("Enter your question: ")

results = retrieve(query, chunks, top_k=2)

for result in results:
    print("\nFile:", result["file"])
    print("Page:", result["page"])
    print(result["text"][:1500])


Enter your question:  what are the key steps in our approach



File: toolformer.pdf
Page: 2
x1: i-1 = Pittsburgh is also known as xi: n = the Steel City x* = Pittsburgh is also known as [QA(What …? → Steel City)] the Steel City. ci1 = What other name is Pittsburgh known by? ci2 = Which country is Pittsburgh in? ri1 = Steel City ri2 = United States Li( ci1 → Steel City ) < min( Li( ci1 → ε), Li(ε)) Li( ci2 → United States ) > min( Li( ci2 → ε), Li(ε))1 Sample API Calls 2 Execute API Calls 3 Filter API Calls LM Dataset LM Dataset with API Calls Figure 2: Key steps in our approach, illustrated for a question answering tool: Given an input text x, we ﬁrst sample a position iand corresponding API call candidates c1 i,c2 i,...,ck i. We then execute these API calls and ﬁlter out all calls which do not reduce the loss Liover the next tokens. All remaining API calls are interleaved with the original text, resulting in a new text x∗. tant not only because of the costs associated with such annotations, but also because what humans ﬁnd useful may be differen